# 第14回：Kaggle改善会

**今日の問い：限られた時間で、次に何を試すか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 限られた時間で実験を優先順位付けし、OOFスタッキングで統合する
- adversarial validationで学習とテストの分布ずれを点検する
- 複数シードの平均と閾値調整で、偶然に頼らない改善を積む

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- OOFスタッキング：OOF予測を入力に上位モデルで統合する方法
- 分布ずれ：学習とテストで入力の分布が違うこと
- シードアンサンブル：乱数だけ変えた複数モデルの平均
- 閾値調整：確率からクラスへの境界を変えること
- 実験統合：有効な変更を再検証しながら組み合わせること

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 改善会：限られた時間で「次の一手」を選ぶ

ベースラインができたら、次は改善です。ただし時間は有限。**闇雲に試すのではなく、分担して1人1変更**を
検証し、良かったものだけを統合します。ここでも第12回の原則（1度に1つ、同じ条件、記録を残す）が効きます。

いちばん大事な心得：**手元の検証（ローカル）とLeaderboardの両方を見る**こと。Leaderboardだけを追うと、
公開スコアに過剰適合して最終順位を落とします。まず、答え合わせ用の`answers`も含めてデータを読みます。


In [ ]:
import pandas as pd
train = pd.read_csv(DATA / "local_competition" / "train.csv")
test = pd.read_csv(DATA / "local_competition" / "test.csv")
answers = pd.read_csv(DATA / "local_competition" / "instructor_answers.csv")


## 5人の担当

1人1テーマに分かれます：**1. 欠損補完 / 2. 特徴量（最適温度からの距離）/ 3. モデルの深さ /
4. 判定閾値 / 5. 誤分類の確認**。全員が同じ`random_state=42`とF1を使い、**担当箇所以外は変えない**——
こうすると「誰の変更が効いたか」を後で切り分けられます。


## 改善案を1つ組んで、ローカルで検証する

この例では2〜3の担当（特徴量追加＋浅い木＋`class_weight`）を1つの案にまとめています。第11回の
`temperature_distance`を足し、第8回の`class_weight="balanced"`で少数クラスを重視。まずローカル検証F1で
ベースラインと比べます。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

improved_train = train.copy()
improved_test = test.copy()
for frame in [improved_train, improved_test]:
    frame["temperature_distance"] = (frame["temperature_c"] - 78).abs()
target = "active"
ignored = ["sample_id", "experiment_date", "smiles", target]
features = [c for c in improved_train.columns if c not in ignored]
numeric = improved_train[features].select_dtypes(include="number").columns.tolist()
categorical = [c for c in features if c not in numeric]
preprocess = ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical),
])
model = Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=42))])
X_train, X_valid, y_train, y_valid = train_test_split(improved_train[features], improved_train[target], test_size=0.25, random_state=42, stratify=improved_train[target])
model.fit(X_train, y_train)
print("改善案のローカルF1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


### 出力の読み方

このローカルF1を、第13回のベースライン（`submission_baseline`を作ったときの検証F1）と比べます。
**上がっていれば採用候補**。ただし1回の分割なので、余裕があれば交差検証（第10回）で確かめると確実です。


## 模擬Leaderboardで答え合わせする

この教材では講師が`answers`（正解）を持っており、ローカルで「提出したつもり」の採点ができます。
全データで学習し直してtestを予測し、`answers`と突き合わせて**模擬Leaderboard F1**を出します。


In [ ]:
model.fit(improved_train[features], improved_train[target])
improved_submission = pd.DataFrame({"sample_id": improved_test["sample_id"], "active": model.predict(improved_test[features])})
merged = answers.merge(improved_submission, on="sample_id", suffixes=("_true", "_pred"))
print("模擬Leaderboard F1:", round(f1_score(merged["active_true"], merged["active_pred"]), 3))


### 出力の読み方と実験ログ

- **ローカルF1と模擬LB F1が近い**なら、手元の検証は信頼できます。**大きく食い違う**なら、過剰適合や分布ずれを疑います。
- 改善しても悪化しても、`変更点 / ローカルF1 / 模擬LB F1 / 気づき`を1行で記録します。
- **Leaderboardだけ上がってローカルが下がった案は要注意**（公開スコアへの過剰適合の疑い）。良い変更だけを慎重に統合します。


## DEEP DIVE：単体を超える3つの技

上位を狙うときの定番を3つ。**OOFスタッキング**（違うモデルを束ねる）、**分布ずれの点検**
（train/testが似ているか）、**シード平均**（乱数の偶然を薄める）です。いずれも第6・10回の応用です。


### OOFスタッキング：違うモデルの予測を束ねる

第10回のスタッキングを、コンペ流に手作りします。3つのモデルの**OOF確率**（第13回）を作り、それらを
入力にした上位モデル（ロジスティック回帰）で統合します。OOFを使うのは、束ねる段階でリークしないためです。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

pre = ColumnTransformer([
    ("n", SimpleImputer(strategy="median"), numeric),
    ("c", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), categorical),
])
members = {
    "rf": make_pipeline(pre, RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42)),
    "hgb": make_pipeline(pre, HistGradientBoostingClassifier(max_iter=200, random_state=42)),
    "logit": make_pipeline(pre, LogisticRegression(max_iter=1000)),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = {}
for name, est in members.items():
    oof[name] = cross_val_predict(est, improved_train[features], improved_train[target], cv=skf, method="predict_proba")[:, 1]
    print(f"{name:6s} OOF F1:", round(f1_score(improved_train[target], (oof[name] >= 0.5).astype(int)), 3))
meta_X = pd.DataFrame(oof)
stack_oof = cross_val_predict(LogisticRegression(max_iter=1000), meta_X, improved_train[target], cv=skf, method="predict_proba")[:, 1]
print("スタッキング OOF F1:", round(f1_score(improved_train[target], (stack_oof >= 0.5).astype(int)), 3))


### 出力の読み方

各モデル単体のOOF F1と、スタッキングのOOF F1を比べます。**スタッキングが単体最良を上回れば**束ねた
価値あり。ほぼ同じなら、モデルたちが似た間違え方をしている（束ねる旨みが少ない）ということ。第10回と
同じ教訓：束ねは万能ではありません。


### 分布ずれを点検する（adversarial validation）

第6回の手法をコンペに適用。trainとtestを見分ける分類器のAUCで、両者の分布の近さを測ります。
AUCが高ければ、ローカル検証がLeaderboardとずれる原因になります。


In [ ]:
from sklearn.model_selection import cross_val_score

combined = pd.concat([
    improved_train[numeric].assign(is_test=0),
    improved_test[numeric].assign(is_test=1),
], ignore_index=True)
adv = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, random_state=42))
auc = cross_val_score(adv, combined[numeric], combined["is_test"], cv=5, scoring="roc_auc")
print("adversarial validation AUC:", round(auc.mean(), 3), "（0.5付近なら分布は近い）")


### 出力の読み方

AUCが0.5付近なら、train/testは似ていて手元CVは信頼できます。高ければ、CV-LBギャップの一因。実データの
コンペでは、AUCを上げている列を特定して扱いを見直す、といった対処につなげます。


### シード平均：乱数の運を薄める

同じモデルでも`random_state`を変えると予測が少し変わります。複数シードの確率を平均すると、**乱数由来の
ばらつきが打ち消し合い**、安定した予測になります。少ない手間で効きやすい定番テクです。


In [ ]:
import numpy as np

probs = []
for seed in [0, 1, 2, 3, 4]:
    est = make_pipeline(pre, RandomForestClassifier(n_estimators=300, max_depth=4, random_state=seed)).fit(improved_train[features], improved_train[target])
    probs.append(est.predict_proba(improved_test[features])[:, 1])
ensemble_pred = (np.mean(probs, axis=0) >= 0.5).astype(int)
seed_merged = answers.merge(pd.DataFrame({"sample_id": improved_test["sample_id"], "active": ensemble_pred}), on="sample_id", suffixes=("_true", "_pred"))
print("5シード平均の模擬LB F1:", round(f1_score(seed_merged["active_true"], seed_merged["active_pred"]), 3))


### 出力の読み方

5シード平均の模擬LB F1が、単一シードのときより**わずかに高く・安定**していれば成功。派手さは
ありませんが、こうした地味で確実な積み上げが、コンペでも実務でも効きます。「1回の高スコア」より
「**再現できる改善**」を選ぶ——この教材全体の締めくくりの姿勢です。


## APPENDIX（任意・追加演習）

改善の詰めを、OOFを使って安全に行います（すべて上のDEEP DIVEで作った`oof`を再利用）。90分の外の
自習向けです。まず**2モデルの重み付き平均（ブレンド）**の最適な重みを、OOF上で探します。


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

y_true = improved_train[target]
best = None
for w in np.linspace(0, 1, 11):
    blend = w * oof["rf"] + (1 - w) * oof["hgb"]
    f1 = f1_score(y_true, (blend >= 0.5).astype(int))
    if best is None or f1 > best[1]:
        best = (round(w, 1), round(f1, 3))
print(f"rf重み={best[0]} のときOOF F1最大={best[1]}")


### 出力の読み方

重み0はhgbのみ、1はrfのみ、途中が混合。**単体より混合が良い重み**が見つかれば、ブレンドの価値あり。
OOFで重みを決めるのは、テストに触れずに（リークなく）調整するためです。ただし重みを探しすぎると
OOFに過剰適合するので、探索は粗め（ここは11点）にとどめます。


### 判定閾値もOOFで最適化する

第8回の閾値調整を、OOF確率に対して行います。0.5に固定せず、F1が最大になる閾値を手元で選びます。


In [ ]:
rows = []
for t in np.linspace(0.2, 0.8, 13):
    rows.append({"閾値": round(t, 2), "F1": f1_score(y_true, (oof["rf"] >= t).astype(int))})
tbl = pd.DataFrame(rows)
print("OOFでF1最大の閾値:", tbl.loc[tbl["F1"].idxmax(), "閾値"])
tbl.round(3)


### 出力の読み方

F1が最大になる閾値が0.5とずれるなら、閾値調整で無料の改善が得られます。**OOFで選んだ閾値を、最終提出に
だけ適用**します（検証に使った同じデータで選んで報告しない、という第6・12回の原則を守ります）。


### 誤分類を群別に分析する

どの化合物系列でよく間違えるかを、OOF予測で集計します。特定の系列に誤りが偏るなら、その系列を
表す特徴量の不足や、データ不足を疑います。次の改善仮説のきっかけになります。


In [ ]:
val = improved_train[["scaffold_group", target]].copy()
val["oof_pred"] = (oof["rf"] >= 0.5).astype(int)
val["誤り"] = val[target] != val["oof_pred"]
by_group = val.groupby("scaffold_group").agg(件数=("誤り", "size"), 誤り数=("誤り", "sum"), 誤り率=("誤り", "mean"))
display(by_group.sort_values("誤り率", ascending=False).round(3))


### 出力の読み方

誤り率の高い系列が、モデルの弱点。件数が十分あるのに誤り率が高い系列は、**その系列に効く特徴量を
足す**（第11回）か、**分割を系列単位にする**（第6回）といった次の一手につながります。エラー分析は、
闇雲なチューニングより効く改善のきっかけです。


## よくある誤り

- 5人の変更を一度に統合する
- Leaderboardだけを目的関数にする
- 分布ずれを無視してランダム分割だけで判断する

## SELF-STUDY（任意・30〜60分）

- 単体最良・投票・スタッキングのOOF F1を比較する
- adversarial validationのAUCが高い列を除いて再評価する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. OOFスタッキングの手順は何か
2. 分布ずれをどう検知するか
3. 改善を統合する順序はどうするか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
